<div align="center">

  <h1><img src="../assets/logo.png" alt="Ion" width="72"><br>Ion Tour Notebook</h1>

</div>


A hands-on walkthrough of [Ion](https://github.com/auxeno/ion), a minimal neural network library for JAX.

By the end of this notebook you'll understand the four core concepts:

1. **`Module`** — define models as frozen JAX pytrees
2. **`Param`** — mark arrays as trainable or frozen
3. **`ion.grad`** — differentiate only trainable parameters
4. **`ion.apply_updates`** — apply optimizer updates to a model

In [ ]:
# !pip install ion-nn optax plotly jaxtyping

In [ ]:
import jax
import jax.numpy as jnp
import optax
import plotly.graph_objects as go

import ion
from ion import nn

## The dataset

We'll learn $\sin(x)$ on $[-\pi, \pi]$ — a simple regression task that runs in seconds on CPU.

In [ ]:
x_train = jnp.linspace(-jnp.pi, jnp.pi, 200).reshape(-1, 1)
y_train = jnp.sin(x_train)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_train[:, 0], y=y_train[:, 0], mode="markers", name="sin(x)", marker=dict(size=6, color="black")))
fig.update_layout(title="Training data", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Defining a Module

Subclass `nn.Module`, declare your fields with type annotations, and write `__init__` and `__call__`. Ion automatically converts your class into a dataclass and registers it as a JAX pytree.

Our MLP stores three kinds of field:
- **`nn.Linear` layers** — sub-modules containing `Param` leaves
- **`activation`** — a callable (e.g. `jax.nn.relu`), wrapped as static metadata automatically
- **`input_scale`** — a plain JAX array that normalises inputs to [-1, 1], but is *not* a `Param` so it won't be trained. Included for demonstration purposes

In [ ]:
from collections.abc import Callable
from jaxtyping import Array, Float, PRNGKeyArray


class MLP(nn.Module):
    fc_1: nn.Linear
    fc_2: nn.Linear
    activation: Callable[[Array], Array]
    input_scale: jax.Array

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, activation=jax.nn.relu, *, key: PRNGKeyArray):
        k1, k2 = jax.random.split(key)
        self.fc_1 = nn.Linear(in_dim, hidden_dim, key=k1)
        self.fc_2 = nn.Linear(hidden_dim, out_dim, key=k2)
        self.activation = activation
        self.input_scale = jnp.array(1.0 / jnp.pi)

    def __call__(self, x: Float[Array, "... d"]) -> Float[Array, "... d"]:
        x = x * self.input_scale
        x = self.activation(self.fc_1(x))
        x = self.fc_2(x)
        return x
    
model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))
model

Treescope is enabled by default for rich interactive rendering. Click on elements in the Treescope HTML to expand/contract them.

Ion also has a built-in text representation for terminals and logging:

In [ ]:
ion.disable_treescope()
print(model)
ion.enable_treescope()

## Params

Every learnable array in a model is wrapped in `nn.Param`. Params track whether they are **trainable** or **frozen**, and they behave transparently as arrays in expressions thanks to `__jax_array__`.

In [ ]:
model.fc_1.w

In [ ]:
# Params work transparently in arithmetic — no need to unwrap
x_sample = jnp.ones((1,))
result = x_sample @ model.fc_1.w

print(f"Result shape: {result.shape}")
print(f"Result type:  {type(result).__name__}  (regular array, not Param)")

`.params` preserves the pytree structure of the model, but sets any non-`Param` leaves to `None`. So it's not just the model parameters, it would be more accurate to say it's the model pytree structure with parameter leaves kept and every other leaf set to None.

In [ ]:
model.params

In [ ]:
print(f"Total parameters: {model.num_params:,}")

## Modules are pytrees

Ion modules are registered as JAX pytrees. This means standard `jax.tree` operations work out of the box — no special API needed.

In [ ]:
# The leaves of the tree are Param values and plain arrays
leaves = jax.tree.leaves(model)
print(f"{len(leaves)} leaves:")
for leaf in leaves:
    print(f"  {str(leaf.dtype):8s} {str(leaf.shape)}")

In [ ]:
# Zero all leaves
jax.tree.map(lambda p: jnp.zeros_like(p), model)

## JAX transforms just work

Because modules are pytrees, `jax.jit` and `jax.vmap` work directly — no wrappers required.

In [ ]:
# JIT-compile the model directly — it's just a callable pytree
jit_model = jax.jit(model)
y_hat = jit_model(x_train[:5])
print(f"JIT output shape: {y_hat.shape}")

In [ ]:
# vmap over a batch dimension — also just works
batched_input = jax.random.normal(jax.random.key(1), (8, 1))
y_batched = jax.vmap(model)(batched_input)
print(f"vmap output shape: {y_batched.shape}")

## Gradients

`ion.grad` and `ion.value_and_grad` work like their JAX counterparts, but they only differentiate **trainable `Param`** leaves. Everything else — frozen params, plain arrays, non-array fields — is held constant.

The returned gradient tree has the same structure as the model, with `None` at every non-trainable position.

In [ ]:
def loss_fn(model, x, y):
    y_hat = model(x)
    return jnp.mean((y_hat - y) ** 2)

loss, grads = ion.value_and_grad(loss_fn)(model, x_train, y_train)
print(f"Loss: {loss:.4f}")

In [ ]:
# The gradient tree mirrors the model structure.
# Params get gradients; non-param leaves like input_scale are None.
# Static metadata like activation is preserved as-is (it's structure, not a leaf).
grads

## Freeze and unfreeze

You can freeze an entire model or individual layers. Frozen parameters have `trainable=False` and are skipped by `ion.grad`. Watch the Treescope `Param` colours change:

In [ ]:
# Freeze the entire model
frozen_model = model.freeze()
frozen_model

In [ ]:
# Selectively freeze just the first layer
partially_frozen = model.replace(fc_1=model.fc_1.freeze())
partially_frozen

In [ ]:
# Gradients respect trainability: frozen layer gets None gradients
frozen_grads = ion.grad(loss_fn)(partially_frozen, x_train, y_train)
frozen_grads

## Training loop

The training loop is straightforward: compute gradients with `ion.value_and_grad`, get updates from an [Optax](https://github.com/google-deepmind/optax) optimizer, and apply them with `ion.apply_updates`.

In [ ]:
# Start fresh
model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))

optimizer = optax.adam(5e-3)
opt_state = optimizer.init(model.params)


@jax.jit
def train_step(model, opt_state, x, y):
    loss, grads = ion.value_and_grad(loss_fn)(model, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, model.params)
    model = ion.apply_updates(model, updates)
    return model, opt_state, loss


losses = []
for step in range(200):
    model, opt_state, loss = train_step(model, opt_state, x_train, y_train)
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.6f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=losses, mode="lines", name="MSE loss", line=dict(width=5)))
fig.update_layout(title="Training loss", xaxis_title="Step", yaxis_title="Loss", height=400, template="plotly_white", yaxis_type="log")
fig.show()

In [ ]:
x_eval = jnp.linspace(-jnp.pi, jnp.pi, 500).reshape(-1, 1)
y_pred = model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=jnp.sin(x_eval[:, 0]), mode="lines", name="sin(x)", line=dict(width=5, color="black")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="Model", line=dict(width=5, dash="dash")))
fig.update_layout(title="Trained model vs sin(x)", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Model surgery

Modules are **frozen** after `__init__` — you can't mutate them in place. To create a modified copy, use `.replace()`. This makes swapping components trivial.

In [ ]:
# Direct mutation is not allowed
try:
    model.fc_1 = nn.Linear(1, 64, key=jax.random.key(99))
except AttributeError as e:
    print(f"AttributeError: {e}")

### Swap activation function

Our `activation` is stored as a regular attribute on the module. We can swap it with `.replace()` — since it's treated as static metadata, `jax.jit` handles the recompilation automatically.

In [ ]:
# Swap activation on the trained model — weights are kept, only activation changes
tanh_model = model.replace(activation=jax.nn.tanh)
opt_state_tanh = optimizer.init(tanh_model.params)

tanh_losses = []
for step in range(200):
    tanh_model, opt_state_tanh, loss = train_step(tanh_model, opt_state_tanh, x_train, y_train)
    tanh_losses.append(loss.item())

print(f"ReLU final loss: {losses[-1]:.6f}")
print(f"Tanh final loss: {tanh_losses[-1]:.6f}")

In [ ]:
y_pred_tanh = tanh_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=jnp.sin(x_eval[:, 0]), mode="lines", name="sin(x)", line=dict(width=5, color="black")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="ReLU", line=dict(width=5)))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_tanh[:, 0], mode="lines", name="Tanh", line=dict(width=5)))
fig.update_layout(title="Activation function comparison", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

### Resize hidden layers

We can swap in wider layers with `.replace()`. The weights are reinitialized since the dimensions change.

In [ ]:
k1, k2 = jax.random.split(jax.random.key(42))
wide_model = model.replace(
    fc_1=nn.Linear(1, 64, key=k1),
    fc_2=nn.Linear(64, 1, key=k2),
)

print(f"Original: {model.num_params:,} params  (hidden_dim=16)")
print(f"Wide:     {wide_model.num_params:,} params  (hidden_dim=64)")

In [ ]:
opt_state_wide = optimizer.init(wide_model.params)

wide_losses = []
for step in range(200):
    wide_model, opt_state_wide, loss = train_step(wide_model, opt_state_wide, x_train, y_train)
    wide_losses.append(loss.item())

y_pred_wide = wide_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=jnp.sin(x_eval[:, 0]), mode="lines", name="sin(x)", line=dict(width=5, color="black")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="16 hidden", line=dict(width=5)))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_wide[:, 0], mode="lines", name="64 hidden", line=dict(width=5)))
fig.update_layout(title="Effect of hidden size", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Checkpointing

`ion.save()` serializes array leaves and trainable flags to a `.npz` file. `ion.load()` restores them into a reference model, which provides the non-array fields like activation functions.

In [ ]:
import tempfile, os

path = os.path.join(tempfile.gettempdir(), "model.npz")
ion.save(path, model)
print(f"Saved to {path}  ({os.path.getsize(path):,} bytes)")

In [ ]:
# Load into a fresh (randomly initialized) model
fresh_model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(999))

print(f"Fresh model loss:  {loss_fn(fresh_model, x_train, y_train):.4f}")

loaded_model = ion.load(path, fresh_model)
print(f"Loaded model loss: {loss_fn(loaded_model, x_train, y_train):.6f}")
print(f"Weights match:     {jnp.allclose(model.fc_1.w.value, loaded_model.fc_1.w.value)}")

In [ ]:
# Trainable flags are preserved through save/load
frozen_model = model.replace(fc_1=model.fc_1.freeze())
ion.save(path, frozen_model)

loaded_frozen = ion.load(path, fresh_model)
print(f"fc_1.w trainable: {loaded_frozen.fc_1.w.trainable}  (was frozen before save)")
print(f"fc_2.w trainable: {loaded_frozen.fc_2.w.trainable}  (was trainable before save)")

os.remove(path)

Check out the [examples](https://github.com/auxeno/ion/tree/main/examples) for full training scripts, or the [docs](https://github.com/auxeno/ion) for more detail.